# Data Manipulation

## Predict Probability

In [ ]:
import pandas as pd
import numpy as np
import os
import csv
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
import torch
from torch.utils.data import Dataset
from tqdm import tqdm

class FileDataset(Dataset):
    def __init__(self, df):
        self.df = df
        super().__init__()

    def __len__(self):
        return len(self.df.index)

    def __getitem__(self, i):
        return self.df.at[i, 'text']

    def getId(self, i):
        return self.df.at[i, 'id']
    
tokenizer = AutoTokenizer.from_pretrained("model", model_max_length=512, padding='max_length', truncation=True,)

model = AutoModelForSequenceClassification.from_pretrained("model")
device = torch.device("cuda")

classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=device,
    top_k=None,
    # Add truncation and padding to the pipeline's parameters
    truncation=True,
    padding='max_length',
    max_length=512,  # Ensure this matches the tokenizer's max length
    return_all_scores=True
)


def logit_forward(input):
    model.eval()
    with torch.no_grad():  # Disable gradient calculations
        # input_ids, attention_mask = inputs
        # print(input)
        # print(type(input))
        # input = list(input)
        tokenized_input = tokenizer(input, padding='max_length', truncation=True, return_tensors="pt", max_length=512)
        # Tokenize the batch
        input_ids = tokenized_input['input_ids'].to(device)
        attention_mask = tokenized_input['attention_mask'].to(device)

        # Make predictions
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits.cpu()

        # Convert logits to predicted labels (e.g., using argmax for classification)
        probs = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()

        return logits


# pos_probs = []
# for i, row_preds in enumerate(tqdm(classifier(dataset, batch_size=32),total=len(dataset))):
#     pos_probs.append(next(item['score'] for item in row_preds if item['label'] == '1'))
#     # print(row_preds)
# df['probability'] = pos_probs
# df.to_csv("dataset_prob.txt", sep="\t", quoting=csv.QUOTE_NONE, index=False)

logits_pos = []
logits_neg = []
df = pd.read_csv('dataset.txt', sep="\t", quoting=csv.QUOTE_NONE)
df = df.drop('logit', axis=1)
display(df.head())
# df = df.head(5)

for text in tqdm(list(df['text'])):
    lo = logit_forward(text)
    logits_pos.append(float(lo[0][1]))
    logits_neg.append(float(lo[0][0]))

df['logit_pos'] = logits_pos
df['logit_neg'] = logits_neg

df.to_csv("dataset.txt", sep="\t", quoting=csv.QUOTE_NONE, index=False)

# Process the data in batches
# texts = list(df['text'])
# for i in tqdm(range(0, len(texts), 32)):
#     batch_texts = texts[i:i+32]  # Extract batch
#     batch_logits = logit_forward(batch_texts)  # Get logits for the batch

#     # Append logits to the respective lists
#     for lo in batch_logits:
#         logits_pos.append(float(lo[1]))
#         logits_neg.append(float(lo[0]))

# # Add the logits to the DataFrame
# df['logit_pos'] = logits_pos
# df['logit_neg'] = logits_neg

# # Save the updated DataFrame
# df.to_csv("dataset.txt", sep="\t", quoting=csv.QUOTE_NONE, index=False)

## Add Probability Groups

In [ ]:
import pandas as pd

def prob_group(prob):
    """
    Takes in a float (prob) between 0 and 1 and returns the group as a string 
    representing the range (e.g., '0.0 to 0.1', '0.1 to 0.2', etc.).
    
    Args:
        prob (float): A probability value between 0 and 1.
        
    Returns:
        str: A string representing the range group.
    """
    if prob < 0 or prob > 1:
        raise ValueError("Input must be a float between 0 and 1.")
    
    # Determine the lower bound of the range
    lower_bound = round(prob // 0.1 * 0.1, 1)
    upper_bound = round(lower_bound + 0.1, 1)
    
    # Format the group as a string
    return f"{lower_bound} to {upper_bound}"


df = pd.read_csv('dataset.txt', sep="\t", quoting=csv.QUOTE_NONE)
df['prob_group'] = df['probability'].apply(prob_group)
df.to_csv("dataset.txt", sep="\t", quoting=csv.QUOTE_NONE, index=False)

print(df['prob_group'].unique())

## Generate Random Sample

In [ ]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv('dataset.txt', sep="\t", quoting=csv.QUOTE_NONE)
# Assuming df is your DataFrame
# Specify the number of rows you want to sample
n_samples = 200

# Perform stratified sampling using groupby
stratified_sample = df.groupby(['dataset', 'prob_group'], group_keys=False).apply(
    lambda x: x.sample(min(len(x), n_samples // len(df.groupby(['dataset', 'prob_group']))), random_state=42))



In [ ]:
from pathlib import Path

# Extract only the "id" values and their corresponding row indexes
sample_ids = stratified_sample[["id"]].reset_index()

sample_data = {
    "id": list(sample_ids["id"])
}
print(sample_data)

# Save sampled IDs to disk.
with Path("sample_ids.json").open("w") as f:
    json.dump(sample_data, f)

# SHAP

In [ ]:
from pathlib import Path
import pandas as pd
import pickle
from tqdm import tqdm
import csv

from multiprocessing import Pool, cpu_count
from workers import shap_worker

sv_results_dir = Path("sv_results")
files = [p.name for p in sv_results_dir.glob("*.pkl")]
files.sort(key=lambda i: int(i.split("_")[2]))

df_sv = pd.DataFrame()
features = []
values = []
base_values = []
ids = []
datasets = []

for file in tqdm(files):
    print(file)
    with (sv_results_dir / file).open("rb") as f:
        sv = pickle.load(f)
    sv_pos = sv[:, :, 1]
    i_min = int(file.split("_")[2])
    i_max = int(file.split("_")[3].replace(".pkl", "")) + 1
    print(i_min, i_max)
    df = pd.read_csv("dataset.txt", sep="\t", quoting=csv.QUOTE_NONE)
    df = df[i_min:i_max]
    df = df.reset_index()
    if len(df) != len(sv_pos):
        raise Exception("length does not match")

    for i in range(len(sv_pos)):
        features.extend(sv_pos.data[i])
        values.extend(sv_pos.values[i])
        ids.extend([df["id"][i]] * len(sv_pos.data[i]))
        datasets.extend([df["dataset"][i]] * len(sv_pos.data[i]))
        base_values.extend([sv_pos.base_values[i]] * len(sv_pos.data[i]))

# with Pool(processes=max(cpu_count() - 1, 1)) as pool:
#     results = tqdm(pool.imap(shap_worker, files), total=len(files))
# for r in results:
#     f, v, b, i, d = r
#     features.extend(f)
#     values.extend(v)
#     ids.extend(i)
#     datasets.extend(d)
#     base_values.extend(b)

df_sv["id"] = ids
df_sv["dataset"] = datasets
df_sv["feature"] = features
df_sv["value"] = values
df_sv["base_value"] = base_values

df_sv.to_csv("shap_feature_attributions.csv", index=False)

  0%|          | 0/31 [00:00<?, ?it/s]

sv_probs_0_1999.pkl
0 2000


  3%|▎         | 1/31 [00:05<02:39,  5.32s/it]

sv_probs_2000_3999.pkl
2000 4000


  6%|▋         | 2/31 [00:09<02:18,  4.78s/it]

...

 94%|█████████▎| 29/31 [02:13<00:09,  4.67s/it]

sv_probs_58000_59999.pkl
58000 60000


 97%|█████████▋| 30/31 [02:17<00:04,  4.53s/it]

sv_probs_60000_60801.pkl
60000 60802


100%|██████████| 31/31 [02:19<00:00,  4.50s/it]


# Intergrated Gradients

In [ ]:
from pathlib import Path
import pandas as pd

ig_results_dir = Path("ig_results")
files = [p.name for p in ig_results_dir.glob("*.csv")]
files.sort(key=lambda i: int(i.split("_")[2]))

combined_data = pd.concat([pd.read_csv(ig_results_dir / file) for file in files])

# Save the combined data to a new CSV file
combined_data.to_csv("ig_feature_attributions.csv", index=False)

# Combine SHAP and IG

In [ ]:
import pandas as pd
import numpy as np
import os
import csv
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
import torch
from torch.utils.data import Dataset
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

df_sv = pd.read_csv('shap_feature_attributions.csv')
df_ig = pd.read_csv('ig_feature_attributions.csv')

df_ig['token_cleaned'] = df_ig['token'].str.replace('##', '', regex=False)
df_sv['feature_cleaned'] = df_sv['feature'].str.lower().str.strip()

print(all(df_sv['id'].unique() == df_ig['id'].unique()))
    

In [ ]:
from workers import combine_shap_ig
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

output = {
    'id': [],
    'dataset': [],
    'sv_value': [],
    'sv_token': [],
    'sv_base_value': [],
    'ig_value': [],
    'ig_token': [],
    'ig_base_value': [],
}

args = []

for id in df_sv['id'].unique():
    args.append((df_sv, df_ig, id))

# if __name__ == "__main__":
#     with Pool(processes=max(cpu_count() - 1, 1)) as pool:
#         results = list(pool.map(combine_shap_ig, args), total=len(args))
#     print('done')
#     for o in results:
#         # print(o)
#         for key in output:
#             output[key].extend(o[key])

    
for id in tqdm(df_sv['id'].unique()):
    df_sv1 = df_sv[df_sv['id'] == id]
    df_ig1 = df_ig[df_ig['id'] == id]

    df_sv1 = df_sv1[1:]
    df_ig1 = df_ig1[~df_ig1['token'].isin(['[PAD]', '[CLS]', '[SEP]'])]

    df_sv1 = df_sv1[: len(df_ig1)]

    output['id'].extend(df_sv1['id'])
    output['dataset'].extend(df_sv1['dataset'])
    output['sv_value'].extend(df_sv1['value'])
    output['sv_token'].extend(df_sv1['feature'])
    output['sv_base_value'].extend(df_sv1['base_value'])
    output['ig_value'].extend(df_ig1['logit'])
    output['ig_token'].extend(df_ig1['token'])
    output['ig_base_value'].extend(df_ig1['baseline_logit'])
    
df_output = pd.DataFrame(output)
df_output.to_csv('feature_attributions.csv', index=False)
    # # Reset indices for comparison
    # tokens = df_ig1['token_cleaned'].reset_index(drop=True)
    # features = df_sv1['feature_cleaned'].reset_index(drop=True)

    # # Identify mismatched rows
    # mismatched_rows = tokens != features

    # # Print mismatched words
    # if any(mismatched_rows):
    #     print("Mismatched words:")
    #     print(df_ig1)
    #     print(df_sv1)
    #     for token, feature in zip(tokens[mismatched_rows], features[mismatched_rows]):
    #         print(f"Token: {token}, Feature: {feature}")
    # else:
    #     print("All words match!")

# GPT

In [ ]:
import pandas as pd

df_fa = pd.read_csv("feature_attributions.csv")
display(df_fa.head())

In [ ]:
display(df_fa.head())
df_fa = df_fa.drop(df_fa.columns[0], axis=1)
df_fa = df_fa.drop(["gpt_value"], axis=1)
display(df_fa.head())

In [ ]:
from pathlib import Path
import json
from tqdm import tqdm
import warnings
import pandas as pd
import numpy as np
from multiprocessing import Pool, cpu_count
import workers

warnings.simplefilter("ignore")

if __name__ == "__main__":
    output = {key: [] for key in list(df_results.columns) + ["gpt_index_value", "gpt_index_word_value"]}

    with Path("info.json").open("r") as file:
        info = json.load(file)

    ids = info["id"]
    dfs_by_id = {key: value for key, value in df_results.groupby("id")}
    args = [(id, dfs_by_id[id], ids) for id in df_results["id"].unique()]

    with Pool(cpu_count() - 2) as pool:
        results = list(tqdm(pool.imap(workers.add_gpt, args), total=len(df_results["id"].unique())))

    for r in results:
        for col in output:
            output[col].extend(r[col])

# for i in tqdm(df_results['id'].unique()):
#     df_i = df_results[df_results['id'] == i]

#     if i not in ids:
#         df_i['gpt_index_value'] = pd.NA
#         df_i['gpt_index_word_value'] = pd.NA

#     else:
#         values = {
#             'gpt_index_value': [],
#             'gpt_index_word_value': []
#         }
#         for directory, column in [("gpt_index_masking", "gpt_index_value"), ("gpt_index_word_masking", "gpt_index_word_value")]:

#             with (Path(directory) / 'results' / f"{i}.json").open('r') as json_file:
#                 r = json.load(json_file)
#                 for index in range(len(df_i)):
#                     try:
#                         values[column].append(float(r[str(index)]))
#                     except:
#                         values[column].append(pd.NA)

#             df_i[column] = values[column]

#     for col in df_i.columns:
#         output[col].extend(list(df_i[col]))

    df_output = pd.DataFrame(output)
    display(df_output.describe())
    display(df_output.head(100))
# df_output.to_csv('combined_result_cleaned.csv')

In [ ]:
df_output.to_csv('feature_attributions.csv', index=False)

# One time use

Combine the SHAP prob values to the df containing shap logit and ig logit

In [ ]:
import pandas as pd
import os
from tqdm import tqdm

df_results = pd.read_csv('results/feature_attributions.csv')
df_new = pd.read_csv("results/feature_attributions_w_sv_probs.csv")

print(len(df_results['id'].unique()))
print(len(df_new['id'].unique()))

C:\Users\zhfwe\AppData\Local\Temp\ipykernel_59288\869943938.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_results = pd.read_csv('results/combined_result_cleaned.csv')
C:\Users\zhfwe\AppData\Local\Temp\ipykernel_59288\869943938.py:6: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_new = pd.read_csv('combined_sv_results.csv')


60802
60802


In [ ]:
dfs = []

for id_ in tqdm(df_results['id'].unique()):
    df1 = df_results[df_results['id'] == id_].copy()
    df2 = df_new[df_new['id'] == id_]

    # slice off the first row of df2, then only up to df1’s length
    df2 = df2.iloc[1:len(df1)+1].copy()

    # reset both indexes so that row 0 lines up with row 0, etc.
    df1.reset_index(drop=True, inplace=True)
    df2.reset_index(drop=True, inplace=True)

    # sanity check
    if not df2['feature'].equals(df1['sv_token']):
        raise Exception(f"Token mismatch for id {id_}")

    # now assign by position
    df1['sv_token_prob'] = df2['feature']
    df1['sv_value_prob'] = df2['value']
    df1['sv_base_value_prob'] = df2['base_value']

    dfs.append(df1)

df_4 = pd.concat(dfs, ignore_index=True)

# df_4 = df_results.merge(
#     df_new[['id', 'feature', 'value', 'base_value']]
#     .rename(columns={
#         'feature':            'sv_token',
#         'value':              'sv_value_prob',
#         'base_value':         'sv_base_value_prob'
#     }),
#     on=['id', 'sv_token'],
#     how='left'
# )

df_4.to_csv("feature_attributions_w_sv_probs.csv")

100%|██████████| 60802/60802 [22:34<00:00, 44.88it/s]


In [ ]:
import pandas as pd
df_results = pd.read_csv('results/feature_attributions.csv')
df_4 = pd.read_csv('results/feature_attributions_w_sv_probs.csv')

print(len(df_4))
print(len(df_results))

display(df_4.head(1000))

C:\Users\zhfwe\AppData\Local\Temp\ipykernel_24128\1467668004.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_results = pd.read_csv('results/combined_result_cleaned.csv')
C:\Users\zhfwe\AppData\Local\Temp\ipykernel_24128\1467668004.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_4 = pd.read_csv('combined_results_cleaned_w_sv_probs.csv')


25266608
25266608


,Unnamed: 0,id,dataset,sv_value,sv_token,sv_base_value,ig_value,ig_token,ig_base_value,gpt_index_value,gpt_index_word_value,sv_token_prob,sv_value_prob,sv_base_value_prob
0,0,29741230,train,0.011411,Randomized,-0.834531,0.000958,randomized,-1.049297,NaN,NaN,Randomized,0.002650,0.302688
1,1,29741230,train,0.014880,Controlled,-0.834531,0.014914,controlled,-1.049297,NaN,NaN,Controlled,0.003494,0.302688
2,2,29741230,train,0.014880,Trial,-0.834531,0.013147,trial,-1.049297,NaN,NaN,Trial,0.003494,0.302688
3,3,29741230,train,0.005706,to,-0.834531,0.017276,to,-1.049297,NaN,NaN,to,0.001368,0.302688
4,4,29741230,train,0.005706,Evaluate,-0.834531,0.002511,evaluate,-1.049297,NaN,NaN,Evaluate,0.001368,0.302688
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,23682110,train,0.010710,fibrosis,-0.856498,-0.009097,fibrosis,-1.049297,NaN,NaN,fibrosis,0.002424,0.298072
996,996,23682110,train,0.005759,of,-0.856498,0.014004,of,-1.049297,NaN,NaN,of,0.001303,0.298072
997,997,23682110,train,0.005759,<,-0.856498,-0.002826,<,-1.049297,NaN,NaN,<,0.001303,0.298072
998,998,23682110,train,0.005759,3,-0.856498,-0.001105,3,-1.049297,NaN,NaN,3,0.001303,0.298072
